In [ ]:
import os
import json
import torch
import pandas as pd
import logging
import importlib
from pathlib import Path
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image as PILImage
from IPython.display import Image
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
import wandb

from pytorch_lightning.callbacks import (
    ModelCheckpoint,
    LearningRateMonitor,
    EarlyStopping,
)
from pytorch_lightning.callbacks import TQDMProgressBar
from pytorch_lightning.utilities import rank_zero_info

import sys
PROJECT_DIR = Path("/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER")
SRC_DIR = str(PROJECT_DIR / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from multiomic_transformer.models import model_lightning
import multiomic_transformer.utils.data_formatter as data_formatter
import multiomic_transformer.utils.experiment_handler as experiment_handler
from multiomic_transformer.models.model_simplified import MultiomicTransformer

random.seed(1337)
np.random.seed(1337)
torch.manual_seed(1337)

torch.set_float32_matmul_precision('medium')

GROUND_TRUTH_DIR = PROJECT_DIR / "data" / "ground_truth_files"
DATA_DIR = Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/PROCESSED_DATA")

In [2]:
DATA_DIR = Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/")

cell_type="mESC"
sample_name="E7.5_rep1"
experiment_name=f"{cell_type}_{sample_name}_tutorial"
organism_code="mm10"

tdf = data_formatter.load_tdf(
    settings_path = DATA_DIR / "PROCESSED_DATA" / experiment_name / "settings.json"
)

# Verify that the data cache files exist. If not, this method will create them.
tdf.create_or_load_data_cache(sample_name=tdf.sample_names[0], force_recalculate=False)

mESC_E7.5_rep1_tutorial: Loaded existing settings
All required files are present.
Skipping data cache creation.


In [3]:
# The ExperimentHandler is a higher level class that handles the training and evaluation of the model.
# It takes in a TrainingDataFormatter object to handle file paths, data loading, and caching.
logging.info("Initializing ExperimentHandler...")
exp = experiment_handler.ExperimentHandler(
    training_data_formatter=tdf,
    experiment_dir=DATA_DIR / "EXPERIMENTS",
    model_num=1,
    silence_warnings=False,
)

Initializing ExperimentHandler...


In [4]:
logging.info("Creating dataset...")
exp.create_multichrom_dataset(max_cached=100)

# Prepares the Train/Val/Test dataloaders, being careful to balance the number of 
# batches from each chromosome in each set.
logging.info("Preparing DataLoader...")
train_loader, val_loader, test_loader = exp.prepare_dataloader(
    batch_size=64,
    num_workers=8
)

# Creates scalers for the RNA and ATAC data based on the training split.
logging.info("Creating scalers...")
exp.create_scalers(train_loader)

exp.print_model_settings()

Creating dataset...
Preparing DataLoader...
Creating scalers...
Model settings for mESC_E7.5_rep1_tutorial:
  Epochs (epochs): 250
  Batch Size (batch_size): 64
  Gradient Accumulation Steps (grad_accum_steps): 1
  Use Gradient Checkpointing (use_grad_ckpt): True
  Model Dimension (d_model): 128
  Number of Heads (num_heads): 4
  Number of Layers (num_layers): 3
  Feed-Forward Dimension (d_ff): 512
  Kernel Size (kernel_size): 64
  Dropout (dropout): 0.1
  Bias Scale (bias_scale): 2.0
  Use Distance Bias (use_dist_bias): True
  Initial Learning Rate (initial_lr): 0.00025


In [5]:

tf_vocab_size = int(exp.dataset.tf_ids.numel())
tg_vocab_size = int(exp.dataset.tg_ids.numel())

model = MultiomicTransformer(
    d_model=exp.d_model,
    num_heads=exp.num_heads,
    num_layers=exp.num_layers,
    d_ff=exp.d_ff,
    dropout=exp.dropout,
    tf_vocab_size=tf_vocab_size,
    tg_vocab_size=tg_vocab_size,
    use_bias=exp.use_dist_bias,
    bias_scale=exp.bias_scale,
    window_pool_size=exp.kernel_size,
)


In [6]:

output_dir = exp.model_training_dir / "lightning_logs"
output_dir.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------
# Callbacks
# -------------------------------------------------------
callbacks = [
    ModelCheckpoint(
        dirpath=output_dir,
        filename="{epoch:02d}-{val/roc_auc:.4f}-{val/loss:.4f}",
        monitor="val/mse_unscaled",
        mode="min",
        save_top_k=3,
        save_last=True,
    ),
    EarlyStopping(
        monitor="val/mse_unscaled",
        mode="min",
        patience=5,
    ),
    LearningRateMonitor(logging_interval="epoch"),
]

In [7]:
# -------------------------------------------------------
# W&B logger
# -------------------------------------------------------
rank_zero_info("Setting up Weights and Biases logger")
wandb_logger = None
wandb_logger_dir = exp.model_training_dir / "wandb_logs"
wandb_logger_dir.mkdir(parents=True, exist_ok=True)

wandb_logger = WandbLogger(
    project="multiomic_transformer",
    name=exp.experiment_name,
    save_dir=wandb_logger_dir,
    log_model=True,
)

# Hide metrics from auto-generated W&B charts
wandb_logger.experiment.define_metric("trainer/global_step", hidden=True)
wandb_logger.experiment.define_metric("epoch", hidden=True)
wandb_logger.experiment.define_metric("lr-AdamW", hidden=True)

rank_zero_info("Setting up PyTorch Lightning Trainer...")
trainer = pl.Trainer(
    max_epochs=exp.epochs,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    strategy="auto",
    logger=wandb_logger,
    callbacks=[TQDMProgressBar(refresh_rate=200)],
    gradient_clip_val=1.0,
    deterministic=True,
    default_root_dir=output_dir,
    enable_progress_bar=True,
    enable_checkpointing=True,
    check_val_every_n_epoch=1,
)

importlib.reload(model_lightning)
lit_model = model_lightning.LitMultiomicTransformer(
    exp=exp,
    model=model
)

trainer.fit(lit_model, train_dataloaders=train_loader, val_dataloaders=val_loader)
wandb.finish()

Setting up Weights and Biases logger
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /gpfs/Home/esm5360/.netrc.
wandb: Currently logged in as: luminarada (luminarada-penn-state-health) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Setting up PyTorch Lightning Trainer...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                 ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model           │ MultiomicTransformer │  1.5 M │ train │     0 │
│ 1 │ val_r2_scaled   │ R2Score              │      0 │ train │     0 │
│ 2 │ val_r2_unscaled │ R2Score              │      0 │ train │     0 │
└───┴─────────────────┴──────────────────────┴────────┴───────┴───────┘

Trainable params: 1.5 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.5 M                                                                                                
Total estimated model params size (MB): 6                                                                          
Modules in train mode: 92                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Sanity Checking: |                                                                                    | 0/? [0…

...

Validation: |                                                                                         | 0/? [0…

Validation: |                                                                                         | 0/? [0…

Validation: |                                                                                         | 0/? [0…

`Trainer.fit` stopped: `max_epochs=250` reached.


epoch,▁▁▂▂▂▂▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇█████
train/avg_step_time_sec,█▃▃▁▃▃▄▄▁▂▂▁▁▃▂▃▂▅▄▃▂▁▄▃▃▇▂▂▇▃▂▂▄▅▃▃▃▃▄▅
train/mse_scaled,█▇▇▆▅▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/step_time_sec,▂▂▁▃▂▃▁▁▅▅▄▃▅▂▄█▁▂▂▂▂▄▂▄▂▂▁▅▃▂▅▁▃▂▂▃▄▆▄▅
trainer/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇███
val/mse_scaled,█▇▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/mse_unscaled,█▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/r2_scaled,▁▆▆▆▆▇▆▇▇▇▇▇▇▇▇▇█▇██████████████████████
val/r2_unscaled,▁▁▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇████████████████████
epoch,249
train/avg_step_time_sec,0.02346


In [8]:
exp.model = lit_model.model

In [ ]:
# Runs gradient attribution to calculate the gradients between each TF input and each TG output.
logging.info("\nRunning Gradient Attribution")
atac_grn_df, batch_atac_grn_df = exp.run_atac_gradient_attribution(
    test_loader,
    chunk_size=1,
    max_batches=None,
    show_tqdm=True,
    save_every_n_batches=100,
    )


Running Gradient Attribution
ATAC Gradient attributions: 100%|████████████████████████████████▊| 233/234 [06:54<00:01,  1.78s/it]


In [ ]:

# Runs gradient attribution to calculate the gradients between each TF input and each TG output.
logging.info("\nRunning Gradient Attribution")
grn_df, batch_grn_df = exp.run_gradient_attribution(
    test_loader,
    chunk_size=1,
    max_batches=None,
    max_tgs_per_batch=None,
    show_tqdm=True,
    save_every_n_batches=100,
    )

In [ ]:
# Loads a ground truth file with columns "Source" and "Target" for TF-TG interactions.
logging.info("Loading ground truth datasets...")
GROUND_TRUTH_DIR = Path(PROJECT_DIR) / "data" / "ground_truth_files"
gt_by_dataset_dict = {
    "ChIP-Atlas mESC": exp.load_ground_truth(GROUND_TRUTH_DIR / "chip_atlas_tf_peak_tg_dist.csv"),
    "RN111": exp.load_ground_truth(GROUND_TRUTH_DIR / "RN111.tsv"),
    "RN112": exp.load_ground_truth(GROUND_TRUTH_DIR / "RN112.tsv"),
    "RN114": exp.load_ground_truth(GROUND_TRUTH_DIR / "RN114.tsv"),
    "RN116": exp.load_ground_truth(GROUND_TRUTH_DIR / "RN116.tsv"),        
},

# Calculates the AUROC of the predicted GRN against multiple ground truth datasets.
logging.info("\nCalculating AUROC")
auroc_df = exp.calculate_auroc_all_sample_gts(exp.grn, gt_by_dataset_dict)     
logging.info(f"Pooled Median AUROC: {auroc_df['pooled_median_auroc'].iloc[0]:.3f}")       
logging.info(f"Per-TF Median AUROC: {auroc_df['per_tf_median_auroc'].iloc[0]:.3f}")

exp.save_handler()